# Compliance Scoring & End-to-End Pipeline Evaluation

Runs a full RFP through the complete pipeline:
1. Parse → Chunk → Upsert to Qdrant + BM25
2. Execute LangGraph swarm (Planner → Writer → Gatekeeper → Judge)
3. Analyze compliance scores, hallucination rates, and missing requirements
4. Query Neo4j compliance matrix for gap analysis

In [ ]:
import sys
import os

# Add ai_engine to path
sys.path.append(os.path.abspath('../ai_engine'))

from agents.state import AgentState
from agents.planner import PlannerAgent
from agents.writer import WriterAgent
from agents.gatekeeper import GatekeeperAgent
from agents.judge import JudgeAgent
from agents.workflow import proposal_swarm_graph
from services.compliance_matrix import ComplianceMatrixService

print("✅ All agents and services imported.")

In [ ]:
# ── Sample RFP Sections for Testing ──

test_sections = [
    {
        "heading_path": ["Technical Requirements", "Cloud Infrastructure"],
        "content": (
            "The vendor shall provide a cloud-based solution hosted on AWS, Azure, or GCP. "
            "The solution must support auto-scaling to handle 10,000 concurrent users. "
            "99.9% uptime SLA is mandatory. All infrastructure must be deployed in "
            "US-East region for data sovereignty compliance."
        ),
    },
    {
        "heading_path": ["Security Requirements", "Data Protection"],
        "content": (
            "All data must be encrypted at rest using AES-256 and in transit using TLS 1.3. "
            "The vendor must hold SOC 2 Type II certification. Annual penetration testing "
            "reports must be submitted. Multi-factor authentication is mandatory for all "
            "administrative access."
        ),
    },
    {
        "heading_path": ["Project Management", "Delivery Timeline"],
        "content": (
            "The project must be delivered in four phases: Discovery (4 weeks), "
            "Implementation (12 weeks), Testing & QA (4 weeks), and Go-Live (2 weeks). "
            "The vendor shall assign a PMP-certified Project Manager. Weekly status "
            "reports are required."
        ),
    }
]

rfp_text = "\n\n".join(
    f"## {' > '.join(s['heading_path'])}\n{s['content']}" for s in test_sections
)

print(f"📋 Test RFP: {len(test_sections)} sections, {len(rfp_text)} chars total.")

In [ ]:
# ── Execute Full LangGraph Swarm Pipeline ──

initial_state: AgentState = {
    "rfp_text": rfp_text,
    "sections": test_sections,
    "plan": {},
    "drafts": {},
    "reviews": [],
    "approved": False,
    "status": "initialized",
}

print("🚀 Running LangGraph swarm pipeline...")
print("   Planner → Writer → Gatekeeper → Judge")
print()

final_state = proposal_swarm_graph.invoke(initial_state)

print(f"\n✅ Pipeline complete.")
print(f"   Drafts generated: {len(final_state.get('drafts', {}))}")
print(f"   Reviews logged:   {len(final_state.get('reviews', []))}")
print(f"   Approved:         {final_state.get('approved', False)}")
print(f"   Final status:     {final_state.get('status', 'unknown')}")

In [ ]:
# ── Analyze Plan Output ──

plan = final_state.get("plan", {})
checklist = plan.get("checklist", [])

print("═" * 60)
print(f"  PLANNER OUTPUT ({len(checklist)} requirements identified)")
print("═" * 60)

for item in checklist:
    print(f"  [{item.get('id', '?'):8s}] {item.get('section', 'Unknown')[:45]}")
    print(f"           Type: {item.get('compliance_type', '-')}")
    print(f"           Length target: {item.get('estimated_length', '-')} words")
    print()

In [ ]:
# ── Analyze Draft Quality ──

drafts = final_state.get("drafts", {})

print("═" * 60)
print(f"  DRAFT ANALYSIS ({len(drafts)} sections written)")
print("═" * 60)

for req_id, draft_text in drafts.items():
    word_count = len(draft_text.split())
    has_citations = "[Source" in draft_text or "[Evidence" in draft_text
    has_placeholders = any(p in draft_text.lower() for p in ["todo", "[insert", "placeholder"])
    
    status = "✅" if not has_placeholders else "⚠️"
    cite_icon = "📎" if has_citations else "❌"
    
    print(f"  {status} [{req_id}] {word_count} words | Citations: {cite_icon}")
    print(f"     Preview: {draft_text[:120]}...")
    print()

In [ ]:
# ── Analyze Judge Scores ──

reviews = final_state.get("reviews", [])

print("═" * 60)
print("  JUDGE SCORING RESULTS")
print("═" * 60)

for review_block in reviews:
    step = review_block.get("step", "unknown")
    
    if step == "gatekeeper":
        passed = review_block.get("passed", False)
        issues = review_block.get("issues_found", 0)
        print(f"\n  🔒 GATEKEEPER: {'PASSED ✅' if passed else 'FAILED ❌'}")
        print(f"     Issues found: {issues}")
        for check in review_block.get("checks", []):
            check_status = '✅' if check.get('passed') else '❌'
            print(f"     {check_status} [{check.get('req_id')}] "
                  f"Score: {check.get('compliance_score', '-')} "
                  f"Rec: {check.get('recommendation', '-')}")
    
    elif step == "llm_judge":
        print(f"\n  ⚖️  LLM JUDGE AGGREGATE:")
        print(f"     Compliance:    {review_block.get('average_compliance', 0):.1f}/10")
        print(f"     Clarity:       {review_block.get('average_clarity', 0):.1f}/10")
        print(f"     Completeness:  {review_block.get('average_completeness', 0):.1f}/10")
        print(f"     Overall:       {review_block.get('overall_score', 0):.1f}/10")
        print(f"     Approved:      {'✅' if review_block.get('passed') else '❌'}")
        
        # Hallucination analysis
        details = review_block.get("details", [])
        hallucinations = [d for d in details if d.get("hallucination_flag")]
        print(f"\n     Hallucination flags: {len(hallucinations)}/{len(details)} sections")
        for h in hallucinations:
            print(f"       ⚠️  [{h.get('req_id')}]: {h.get('unsupported_claims', [])}")

print("\n" + "═" * 60)

In [ ]:
# ── Neo4j Compliance Matrix Check ──
# This cell requires a running Neo4j instance

try:
    compliance = ComplianceMatrixService()
    
    # Get summary stats
    summary = compliance.get_compliance_summary()
    
    print("═" * 60)
    print("  NEO4J COMPLIANCE MATRIX SUMMARY")
    print("═" * 60)
    print(f"  Total requirements:  {summary['total_requirements']}")
    print(f"  Compliant:           {summary['compliant']}")
    print(f"  Partial:             {summary['partial']}")
    print(f"  Non-compliant:       {summary['non_compliant']}")
    print(f"  Missing/Unlinked:    {summary['missing']}")
    print(f"  Compliance Rate:     {summary['compliance_rate']}%")
    print("═" * 60)
    
    # Show gaps
    gaps = compliance.get_missing_requirements()
    if gaps:
        print(f"\n  ⚠️  {len(gaps)} UNADDRESSED REQUIREMENTS:")
        for gap in gaps[:10]:
            mandatory = "🔴 MANDATORY" if gap.get("is_mandatory") else "🟡 Optional"
            print(f"     {mandatory} [{gap['requirement_id']}] {gap.get('description', '')[:60]}")
    
    compliance.close()

except Exception as e:
    print(f"⚠️  Neo4j not available: {e}")
    print("   Skipping compliance matrix check. Start Neo4j to enable this cell.")